# 🧹 HR Analytics Project - Notebook 1: Data Cleaning

**Project:** HR Analytics: Workforce Intelligence Dashboard  
**Author:** Alex Lim  
**Last Updated:** 31 Jan. 2026

---

## 📌 Purpose of This Notebook

This notebook documents the full data cleaning process for our synthetic HR dataset.
It is intentionally written step-by-step so the logic is transparent and reproducible.

The raw dataset (`hr_data_raw.csv`) was designed with realistic data quality issues
that mirror what HR professionals encounter in real HRIS systems:

| Issue | Example |
|-------|---------|
| Inconsistent casing | `MALE`, `male`, `M`, `m` |
| Mixed date formats | `2021-03-15`, `15/03/2021`, `March 15 2021` |
| Department name variants | `HR`, `H.R.`, `Human Resources`, `hr` |
| Salary formatting | `$5,000`, `5000`, `5,000` |
| Rating formats | `4` vs `4/5` |
| Missing values | blanks, `N/A`, `-` mixed together |

---

## 🗂️ Notebook Structure

1. Setup & Load Data
2. First Look - Understanding the Raw Data
3. Step-by-Step Cleaning
4. Creating Calculated Columns
5. Saving the Clean Dataset
6. Summary & What's Next

---
## 1. Setup - Import Libraries

We only need two libraries for cleaning:
- **pandas** - the main tool for working with tabular data in Python
- **numpy** - for handling missing values (`NaN`) and numeric operations

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Fixed reference date and keeps age/tenure calculations reproducible
TODAY = datetime(2025, 1, 1)

print('✅ Libraries loaded successfully')

✅ Libraries loaded successfully


---
## 2. First Look - Understanding the Raw Data

Before cleaning anything, we always **explore the raw data first**.
This tells us what problems exist and helps us plan our cleaning steps.

> 💡 **Golden Rule:** Never edit the raw file directly. Always work on a copy.

In [2]:
# Load the raw dataset
# Note: adjust the path if you're running from a different folder
df_raw = pd.read_csv('../data/raw/hr_data_raw.csv')

print(f'Shape: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns')
df_raw.head(10)

Shape: 1000 rows × 36 columns


,Employee_ID,Full_Name,Gender,Date_of_Birth,Nationality,Region,Country,Office_Location,Department,Job_Title,...,Last_Salary_Increase_Pct,Bonus_Received,Training_Hours_Year,Courses_Completed,Last_Training_Date,LD_Budget_Used_Pct,Recruitment_Source,Time_to_Hire_Days,Offer_Accepted,Interviewer_Rating
0,EMP0001,TAN CHEN,Female,26/12/1980,Malaysian,Asia Pacific,Singapore,Singapore,FINANCE,Finance Manager,...,3.0%,Yes,45.4,4.0,July 08 2024,63.0,LinkedIn,35.0,Yes,3.0
1,EMP0002,Arun Ng,Male,August 31 1999,Malaysian,Asia Pacific,Singapore,Singapore,Sales,Account Executive,...,5.3,No,57.3,11.0,27/05/2022,65.0,Referral,46.0,Yes,5.0
2,EMP0003,lee williams,Male,January 23 1981,Korean,Asia Pacific,Singapore,Singapore,Marketing,Marketing Specialist,...,8.9%,Yes,28.1,8.0,2021-01-24,9.0,Internal Transfer,58.0,Yes,4.0
3,EMP0004,Amanda Kumar,Female,1970-11-16,Australian,Asia Pacific,Singapore,Singapore,Legal,Legal Manager,...,4.0,No,12.7,3.0,2023-03-29,52.0,Job Board,72.0,Yes,3.0
4,EMP0005,Arun Brown,female,1988-04-05,Malaysian,Asia Pacific,Singapore,Singapore,HR,HR Assistant,...,1.5,N,29.8,2.0,2023-05-25,27.0,Internal Transfer,69.0,Yes,5.0
5,EMP0006,Sophie Le,NB,1999-09-10,Korean,Asia Pacific,Singapore,Singapore,IT,Software Engineer,...,8.8,Yes,68.8,3.0,April 27 2022,96.0,LinkedIn,23.0,Yes,3.0
6,EMP0007,RAVI WANG,Not Declared,1995-04-04,Australian,Asia Pacific,Singapore,Singapore,HR Dept,HR Specialist,...,0.1%,NO,39.8,2.0,August 08 2021,12.0,AGENCY,58.0,Yes,3.0
7,EMP0008,sarah smith,Female,May 11 1986,American,Asia Pacific,Singapore,Singapore,Sales,Regional Sales Manager,...,13.1,No,61.0,4.0,29/11/2022,98.0,Referral,18.0,Yes,4.0
8,EMP0009,Min Singh,Female,September 22 1984,Thai,Asia Pacific,Singapore,Singapore,Sales,Sales Executive,...,1.6,Yes,18.8,8.0,NaN,15.0,LinkedIn,15.0,Yes,4.0
9,EMP0010,michelle tan,Female,2000-05-20,Korean,Asia Pacific,Singapore,Singapore,Finance,Senior Finance Analyst,...,4.8,N,55.3,9.0,27/08/2024,81.0,Referral,51.0,yes,3.0


In [3]:
# Check data types, notice how dates and numbers are stored as 'object' (text)
# This is a common problem in real HR data exports
print('Column data types (before cleaning):')
print(df_raw.dtypes)

Column data types (before cleaning):
Employee_ID                  object
Full_Name                    object
Gender                       object
Date_of_Birth                object
Nationality                  object
Region                       object
Country                      object
Office_Location              object
Department                   object
Job_Title                    object
Grade_Level                  object
Work_Mode                    object
Hire_Date                    object
Exit_Date                    object
Employment_Status            object
Employment_Type              object
Probation_Passed             object
Termination_Category         object
Termination_Reason           object
Currency                     object
FX_Rate_to_USD              float64
Monthly_Salary_Local         object
Monthly_CTC_Local            object
Monthly_Salary_USD            int64
Monthly_CTC_USD               int64
Performance_Rating           object
Last_Salary_Increase_Pct   

In [4]:
# Check missing values across all columns
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(1)

missing_summary = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).query('`Missing Count` > 0').sort_values('Missing Count', ascending=False)

print(f'Columns with missing values: {len(missing_summary)}')
missing_summary

Columns with missing values: 9


,Missing Count,Missing %
Exit_Date,846,84.6
Termination_Category,796,79.6
Termination_Reason,796,79.6
Last_Training_Date,50,5.0
Time_to_Hire_Days,43,4.3
Interviewer_Rating,43,4.3
Training_Hours_Year,40,4.0
Courses_Completed,32,3.2
LD_Budget_Used_Pct,30,3.0


In [5]:
# Preview the messy columns, see the inconsistencies clearly
print('=== Department variants ===')
print(df_raw['Department'].value_counts().head(15))

print('\n=== Gender variants ===')
print(df_raw['Gender'].value_counts())

print('\n=== Date format samples (Hire_Date) ===')
print(df_raw['Hire_Date'].sample(10, random_state=1).tolist())

print('\n=== Salary format samples ===')
print(df_raw['Monthly_Salary_Local'].sample(10, random_state=1).tolist())

=== Department variants ===
Department
Sales        136
Delivery     116
Legal        112
HR           112
Marketing    110
Finance      109
IT            95
legal         14
sales         14
FINANCE       12
Lgl           12
hr            11
I.T.          11
Mktg          10
Fin           10
Name: count, dtype: int64

=== Gender variants ===
Gender
Female          320
Male            296
Not Declared    177
M                26
non-binary       21
MALE             21
m                21
f                20
FEMALE           20
F                17
female           16
NB               16
Non Binary       15
male             14
Name: count, dtype: int64

=== Date format samples (Hire_Date) ===
['2022-06-02', '26/09/2022', '2024-03-10', '2023-03-15', '2019-06-05', '2017-02-19', '16/11/2024', '2022-08-18', '09/05/2023', '23/06/2024']

=== Salary format samples ===
['119537', '57581', '7435', '164,198', '20521', '$15,000,000', '6181', '82878', '178652', '6,922']


---
## 3. Step-by-Step Cleaning

Now we know what's wrong, we fix it systematically.
We always work on `df` (a copy of the raw data), never on `df_raw`.

### Why Python instead of Excel for this?
Excel is great for a quick visual scan, but Python is better here because:
- We have 1,000 rows across 36 columns - manual Excel fixes would take hours
- Python cleaning is **documented, repeatable, and auditable**
- When the dataset is refreshed, we just re-run the script hence no manual rework

In [6]:
# Always work on a copy - never touch the raw data
df = df_raw.copy()
print(f'✅ Working copy created: {df.shape}')

✅ Working copy created: (1000, 36)


### Step 3a - Standardise Text Columns

We use a **mapping dictionary** a lookup table that says
_"if you see this messy value, replace it with this clean value"_.

In [7]:
# ── Full Name: Proper Title Case ──────────────────────────────
df['Full_Name'] = df['Full_Name'].str.strip().str.title()

# ── Gender: Map all variants to 3 clean values ────────────────
gender_map = {
    'male': 'Male', 'm': 'Male', 'MALE': 'Male', 'M': 'Male',
    'female': 'Female', 'f': 'Female', 'FEMALE': 'Female', 'F': 'Female',
    'non-binary': 'Non-binary', 'non binary': 'Non-binary',
    'NB': 'Non-binary', 'nb': 'Non-binary', 'Non Binary': 'Non-binary',
}
df['Gender'] = df['Gender'].str.strip().replace(gender_map)

# ── Department: Map all variants to 7 clean values ───────────
dept_map = {
    'hr': 'HR', 'h.r.': 'HR', 'human resources': 'HR', 'hr dept': 'HR',
    'it': 'IT', 'i.t.': 'IT', 'information technology': 'IT', 'tech': 'IT',
    'finance': 'Finance', 'FINANCE': 'Finance', 'fin': 'Finance',
    'legal': 'Legal', 'LEGAL': 'Legal', 'lgl': 'Legal',
    'marketing': 'Marketing', 'MARKETING': 'Marketing', 'mktg': 'Marketing',
    'delivery': 'Delivery', 'DELIVERY': 'Delivery', 'del': 'Delivery',
    'sales': 'Sales', 'SALES': 'Sales', 'sls': 'Sales',
}
df['Department'] = df['Department'].str.strip().str.lower().replace(dept_map)
df['Department'] = df['Department'].str.title().replace({'Hr': 'HR', 'It': 'IT'})

# ── Employment Type ───────────────────────────────────────────
emptype_map = {
    'full time': 'Full-time', 'ft': 'Full-time', 'fulltime': 'Full-time',
    'full-time': 'Full-time', 'contractor': 'Contractor',
    'contract': 'Contractor', 'ctr': 'Contractor',
    'intern': 'Intern', 'internship': 'Intern', 'INTERN': 'Intern',
    'expat': 'Expat', 'expatriate': 'Expat', 'EXPAT': 'Expat',
}
df['Employment_Type'] = df['Employment_Type'].str.strip().str.lower().replace(emptype_map)
df['Employment_Type'] = df['Employment_Type'].str.strip().str.title()

# ── Recruitment Source ────────────────────────────────────────
source_map = {
    'linkedin': 'LinkedIn', 'linked in': 'LinkedIn', 'LINKEDIN': 'LinkedIn',
    'referral': 'Referral', 'REFERRAL': 'Referral', 'ref': 'Referral',
    'agency': 'Agency', 'AGENCY': 'Agency', 'recruitment agency': 'Agency',
    'job board': 'Job Board', 'JOB BOARD': 'Job Board', 'jobboard': 'Job Board',
    'internal transfer': 'Internal Transfer', 'internal': 'Internal Transfer',
    'int transfer': 'Internal Transfer',
}
df['Recruitment_Source'] = df['Recruitment_Source'].str.strip().str.lower().replace(source_map)
df['Recruitment_Source'] = df['Recruitment_Source'].str.strip().str.title()

# ── Verify results ────────────────────────────────────────────
print('Gender unique values:     ', sorted(df['Gender'].unique()))
print('Department unique values: ', sorted(df['Department'].unique()))
print('Employment_Type values:   ', sorted(df['Employment_Type'].unique()))

Gender unique values:      ['Female', 'Male', 'Non-binary', 'Not Declared']
Department unique values:  ['Delivery', 'Finance', 'HR', 'IT', 'Legal', 'Marketing', 'Sales']
Employment_Type values:    ['Contractor', 'Expat', 'Full-Time', 'Intern']


### Step 3b - Parse Date Columns

We have three different date formats mixed together.
We write a helper function that tries each format and returns a proper `datetime` object.

In [8]:
def parse_date(value):
    """Try multiple date formats and return NaT if none match."""
    if pd.isna(value) or str(value).strip() in ['', 'N/A', '-', 'nan']:
        return pd.NaT
    v = str(value).strip()
    for fmt in ['%Y-%m-%d', '%d/%m/%Y', '%B %d %Y', '%b %d %Y']:
        try:
            return datetime.strptime(v, fmt)
        except:
            continue
    return pd.NaT

date_cols = ['Date_of_Birth', 'Hire_Date', 'Exit_Date', 'Last_Training_Date']
for col in date_cols:
    df[col] = df[col].apply(parse_date)
    parsed  = df[col].notna().sum()
    missing = df[col].isna().sum()
    print(f'{col:<25} {parsed:>4} parsed  |  {missing:>4} blank/NaT')

Date_of_Birth             1000 parsed  |     0 blank/NaT
Hire_Date                 1000 parsed  |     0 blank/NaT
Exit_Date                  106 parsed  |   894 blank/NaT
Last_Training_Date         950 parsed  |    50 blank/NaT


### Step 3c - Clean Numeric Columns

Some salary values have `$` signs and commas. Some ratings are `"3/5"` instead of `3`.
We strip all formatting characters and convert to proper numbers.

In [9]:
def clean_numeric(value):
    """Remove $, commas, spaces → return float."""
    if pd.isna(value): return np.nan
    return pd.to_numeric(str(value).replace('$','').replace(',','').strip(),
                         errors='coerce')

def clean_rating(value):
    """Handle '3/5' format → return 3.0"""
    if pd.isna(value): return np.nan
    v = str(value).strip()
    return pd.to_numeric(v.split('/')[0].strip(), errors='coerce')

def clean_pct(value):
    """Remove % sign → return float"""
    if pd.isna(value): return np.nan
    return pd.to_numeric(str(value).replace('%','').strip(), errors='coerce')

# Apply each cleaner to the right column
df['Monthly_Salary_Local']    = df['Monthly_Salary_Local'].apply(clean_numeric)
df['Monthly_CTC_Local']       = df['Monthly_CTC_Local'].apply(clean_numeric)
df['Performance_Rating']      = df['Performance_Rating'].apply(clean_rating)
df['Last_Salary_Increase_Pct']= df['Last_Salary_Increase_Pct'].apply(clean_pct)

print('Sample Monthly_Salary_Local values after cleaning:')
print(df['Monthly_Salary_Local'].head(5).tolist())
print('\nPerformance_Rating range:', df['Performance_Rating'].min(),
      '–', df['Performance_Rating'].max())

Sample Monthly_Salary_Local values after cleaning:
[12937, 7900, 9236, 15333, 5352]

Performance_Rating range: 1 – 5


### Step 3d - Standardise Yes/No Columns

Three columns have messy boolean values like `Y`, `yes`, `YES`, `y`.
We map everything to clean `Yes` / `No`.

In [10]:
yes_vals = {'yes', 'y', '1', 'true', 'Yes', 'YES', 'Y'}
no_vals  = {'no',  'n', '0', 'false', 'No', 'NO',  'N'}

def map_yesno(val):
    if pd.isna(val): return np.nan
    v = str(val).strip()
    if v in yes_vals: return 'Yes'
    if v in no_vals:  return 'No'
    return val

for col in ['Probation_Passed', 'Bonus_Received', 'Offer_Accepted']:
    df[col] = df[col].apply(map_yesno)
    print(f'{col:<20} → unique values: {sorted(df[col].dropna().unique())}')

Probation_Passed     → unique values: ['No', 'Yes']
Bonus_Received       → unique values: ['No', 'Yes']
Offer_Accepted       → unique values: ['No', 'Yes']


### Step 3e - Handle Missing Values

Different columns need different strategies:

| Column | Strategy | Reason |
|--------|----------|--------|
| `Termination_Category/Reason` | Fill with `'Not Applicable'` | Active employees didn't leave, blank is misleading |
| `Exit_Date` | Leave as `NaT` | Blank correctly means still employed |
| Numeric training/recruitment fields | Leave as `NaN` | We don't invent data we don't have |

In [11]:
# Termination fields are fill blanks so Tableau filters work cleanly
for col in ['Termination_Category', 'Termination_Reason']:
    before = df[col].isna().sum()
    df[col] = df[col].fillna('Not Applicable')
    print(f'{col}: {before} blanks filled with "Not Applicable"')

# Everything else: leave NaN as-is (honest about unknowns)
print('\nRemaining NaN counts (these are intentionally kept):')
nulls = df.isnull().sum()
print(nulls[nulls > 0].to_string())

Termination_Category: 796 blanks filled with "Not Applicable"
Termination_Reason: 796 blanks filled with "Not Applicable"

Remaining NaN counts (these are intentionally kept):
Exit_Date              894
Training_Hours_Year     40
Courses_Completed       32
Last_Training_Date      50
LD_Budget_Used_Pct      30
Time_to_Hire_Days       43
Interviewer_Rating      43


---
## 4. Creating Calculated Columns

We now **derive new columns** from existing ones.
These calculated fields are extremely useful in Tableau and for EDA.

| New Column | Formula | Purpose |
|------------|---------|----------|
| `Age` | Today − Date_of_Birth | Workforce age analysis |
| `Tenure_Years` | Exit/Today − Hire_Date | Retention analysis |
| `Annual_Salary_USD` | Monthly_Salary_USD × 12 | Annual cost view |
| `Annual_CTC_USD` | Monthly_CTC_USD × 12 | Total employer cost |
| `Attrition_Flag` | 1 if resigned/terminated | Attrition rate calculation |

In [12]:
# Age (in full years)
df['Age'] = df['Date_of_Birth'].apply(
    lambda dob: (TODAY - dob).days // 365 if pd.notna(dob) else np.nan
)

# Tenure (years) uses Exit_Date for leavers, TODAY for active employees
def calc_tenure(row):
    if pd.isna(row['Hire_Date']): return np.nan
    end = row['Exit_Date'] if pd.notna(row['Exit_Date']) else TODAY
    return round((end - row['Hire_Date']).days / 365.25, 1)

df['Tenure_Years'] = df.apply(calc_tenure, axis=1)

# Annual salary figures
df['Annual_Salary_USD'] = (df['Monthly_Salary_USD'] * 12).round(0)
df['Annual_CTC_USD']    = (df['Monthly_CTC_USD']    * 12).round(0)

# Attrition flag (1 = left the company)
df['Attrition_Flag'] = df['Employment_Status'].apply(
    lambda s: 1 if s in ['Resigned', 'Terminated'] else 0
)

attrition_rate = df['Attrition_Flag'].mean() * 100

print(f'Age range      : {int(df["Age"].min())} – {int(df["Age"].max())} years')
print(f'Tenure range   : {df["Tenure_Years"].min()} – {df["Tenure_Years"].max()} years')
print(f'Attrition rate : {attrition_rate:.1f}%')
print(f'New columns    : Age, Tenure_Years, Annual_Salary_USD, Annual_CTC_USD, Attrition_Flag')

Age range      : 23 – 54 years
Tenure range   : 0.0 – 10.0 years
Attrition rate : 20.4%
New columns    : Age, Tenure_Years, Annual_Salary_USD, Annual_CTC_USD, Attrition_Flag


---
## 5. Save the Clean Dataset

We export the clean data as a new CSV. The raw file is never touched.

In [13]:
# Format dates as strings for clean CSV export
for col in ['Date_of_Birth', 'Hire_Date', 'Exit_Date', 'Last_Training_Date']:
    df[col] = df[col].dt.strftime('%Y-%m-%d')

output_path = '../data/cleaned/hr_data_cleaned.csv'
df.to_csv(output_path, index=False)

print(f'✅ Clean dataset saved to: {output_path}')
print(f'   Final shape: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'\nColumn list:')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:>2}. {col}')

✅ Clean dataset saved to: ../data/cleaned/hr_data_cleaned.csv
   Final shape: 1000 rows × 41 columns

Column list:
   1. Employee_ID
   2. Full_Name
   3. Gender
   4. Date_of_Birth
   5. Nationality
   6. Region
   7. Country
   8. Office_Location
   9. Department
  10. Job_Title
  11. Grade_Level
  12. Work_Mode
  13. Hire_Date
  14. Exit_Date
  15. Employment_Status
  16. Employment_Type
  17. Probation_Passed
  18. Termination_Category
  19. Termination_Reason
  20. Currency
  21. FX_Rate_to_USD
  22. Monthly_Salary_Local
  23. Monthly_CTC_Local
  24. Monthly_Salary_USD
  25. Monthly_CTC_USD
  26. Performance_Rating
  27. Last_Salary_Increase_Pct
  28. Bonus_Received
  29. Training_Hours_Year
  30. Courses_Completed
  31. Last_Training_Date
  32. LD_Budget_Used_Pct
  33. Recruitment_Source
  34. Time_to_Hire_Days
  35. Offer_Accepted
  36. Interviewer_Rating
  37. Age
  38. Tenure_Years
  39. Annual_Salary_USD
  40. Annual_CTC_USD
  41. Attrition_Flag


---
## 6. Before vs After - Cleaning Summary

Let's do a final side-by-side comparison to confirm all cleaning worked.

In [14]:
print('=' * 55)
print(f'{"Metric":<35} {"Before":>8} {"After":>8}')
print('-' * 55)
print(f'{"Total rows":<35} {len(df_raw):>8} {len(df):>8}')
print(f'{"Total columns":<35} {df_raw.shape[1]:>8} {df.shape[1]:>8}')
print(f'{"Unique Department values":<35} {df_raw["Department"].nunique():>8} {df["Department"].nunique():>8}')
print(f'{"Unique Gender values":<35} {df_raw["Gender"].nunique():>8} {df["Gender"].nunique():>8}')
print(f'{"Total missing values":<35} {df_raw.isnull().sum().sum():>8} {df.isnull().sum().sum():>8}')
print('=' * 55)
print('\n✅ Cleaning complete! Proceed to Notebook 02 for EDA.')

Metric                                Before    After
-------------------------------------------------------
Total rows                              1000     1000
Total columns                             36       41
Unique Department values                  30        7
Unique Gender values                      14        4
Total missing values                    2676     1132

✅ Cleaning complete! Proceed to Notebook 02 for EDA.


---
## ✅ Summary

In this notebook we:

- Loaded and inspected 1,000 rows of raw, messy HR data
- Standardised 6 text columns using mapping dictionaries
- Parsed 4 date columns from 3 different formats into a standard `YYYY-MM-DD`
- Cleaned salary, rating, and percentage columns by removing formatting characters
- Standardised 3 Yes/No columns from multiple variants
- Made deliberate, documented decisions about how to handle missing values
- Created 5 new calculated columns (Age, Tenure, Annual salary, Attrition Flag)
- Saved the clean dataset ready for analysis

### Key Principle
> Every cleaning decision is explicit and documented here. Anyone can read this notebook and understand exactly what was done to the data and why, this is what makes analysis trustworthy.

---
**➡️ Next: [Notebook 02 - Exploratory Data Analysis](02_eda.ipynb)**